In [1]:
"""
Two-instrument return forecasting solution — Ridge-only, fold-4 fixed.

Public API:
    parameters = modelEstimate(trainingFilename)
    predictions = modelForecast(testFilename, parameters)

Models:
    - Ridge        (active)
    - Lasso        (available, disabled)
    - Ridge_w      : Ridge trained on winsorised target (available, disabled)
    - Lasso_w      : Lasso trained on winsorised target (available, disabled)
    - Huber        (available, disabled)

Model inclusion / tuning:
    Configure MODEL_PIPELINE_CONFIG near the top of the script.

    For each model:
        include = True/False
            Whether the model is included in CV, final training, reporting, and forecasting.

        tune = True/False
            Whether hyperparameter tuning is allowed for that model.

    ENABLE_HYPERPARAMETER_TUNING is still a global master switch.
        A model is tuned only if:
            ENABLE_HYPERPARAMETER_TUNING is True
            and MODEL_PIPELINE_CONFIG[model]["tune"] is True

    If tuning is enabled for a model:
        The model is evaluated in preset/base form and tuned form.
        The family switches to tuned hyperparameters only if tuned mean validation
        improvement vs the training-mean baseline is strictly higher than preset.

============================================================
FOLD-4 FIX (2013-08-19 .. 2013-09-16 validation window)
============================================================
Diagnosis:
    With the previous configuration (8 features, alpha=50), fold 4 validated
    NEGATIVE vs the training-mean baseline (-0.67%) while folds 1/2/3/5 were
    +1.0 .. +1.7%. Decomposition of the fold-4 MSPE showed:
      * corr(prediction, target) collapsed to ~0.02 in the fold-4 window
        (vs 0.10-0.13 in the other folds) while the prediction std stayed at
        its usual level (~0.014). Whenever var(pred) > 2*cov(pred, y), a model
        loses to the mean baseline — exactly what happened.
      * The damage was systematic (11 of 19 validation days negative), and
        concentrated on LOW-volatility days: the fold-4 window is a quiet
        late-August/early-September regime (baseline MSPE 0.017 vs 0.036 in
        fold 1).
      * Two features carried most of the fragility:
          - xret_price_dev_6: its validation-window correlation with the
            target decayed to ~0.015 in fold 4; subset searches at several
            alphas never placed it in any of the top all-positive subsets.
          - yret_price_dev_60 and ret_factor_zscore_60_yw_2_55 are highly
            collinear (train corr ~0.89). OLS-like Ridge (alpha=50 is
            negligible at ~200k rows) exploited their small DIFFERENCE with
            large opposite-signed coefficients; that ill-conditioned contrast
            flipped sign in the fold-4 window and actively hurt.

Fix (three coordinated changes, selected on the same 5 CV folds):
    1. FEATURE_COLS drops xret_price_dev_6 (7 features remain).
    2. Standardized features are clipped to +/- _FEATURE_CLIP_Z (=2.5) before
       fitting and predicting. Clipping caps the leverage of heavy-tailed
       feature outliers; on its own it lifts folds 1/2/3/5 substantially.
    3. _RIDGE_ALPHA raised 50 -> 9600. With ~200k standardized rows the
       penalty mostly shrinks LOW-variance eigen-directions, i.e. precisely
       the unstable contrast between the collinear pair, while leaving the
       well-identified directions nearly untouched.

    Per-fold validation improvement vs baseline, before -> after:
        fold 1: +1.02% -> +0.55%
        fold 2: +1.41% -> +0.99%
        fold 3: +1.39% -> +1.75%
        fold 4: -0.67% -> +0.16%   (now positive)
        fold 5: +1.69% -> +1.58%
        mean  : +0.97% -> +1.01%   (all 5 folds positive)
    The configuration sits on a plateau: every neighbouring setting
    (clip 2.5-3.0, alpha 6400-12800, with or without also dropping
    yret_price_dev_360_rsi_360) is also all-positive.

Approaches tested that did NOT fix fold 4 (kept out of the script):
    * Recency weighting of training rows and rolling (windowed) training —
      both made fold 4 worse; the problem is regime fragility, not stale data.
    * Inverse-variance (per-day GLS) sample weights — worse on fold 4.
    * Raising alpha alone (no clip, all 8 features) — needs alpha ~25600 to
      turn fold 4 barely positive and gives up most of the mean gain.
    * Orthogonalizing the collinear pair — a pure reparameterisation; with a
      near-zero effective penalty it cannot change predictions.
    * Causal volatility scaling of predictions — once the volatility
      normaliser is computed consistently on the training window, it does not
      rescue fold 4.

Causality guarantee for RSI features:
    - RSI warm-up uses only observations available through the current row.
    - No future rows are used to initialise RSI.

Optimisations included:
    - Cached CV folds: feature scaling and fold arrays are built once.
    - joblib parallelism across CV folds / grid candidates where available.
    - Forecasting uses saved linear coefficients for all models.
"""

from __future__ import annotations

from time import perf_counter
from typing import Any, Optional

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, HuberRegressor

try:
    from joblib import Parallel, delayed
except Exception:  # pragma: no cover
    Parallel = None
    delayed = None


# ============================================================
# User defaults / globals
# ============================================================

# Global master switch.
# A model is tuned only if this is True AND that model's config below has "tune": True.
ENABLE_HYPERPARAMETER_TUNING = False

# Per-model pipeline controls.
#
# include:
#     True  -> include the model in CV, training, reporting, and forecasting.
#     False -> completely exclude the model from the pipeline.
#
# tune:
#     True  -> allow hyperparameter tuning for this model, subject to the global
#              ENABLE_HYPERPARAMETER_TUNING switch.
#     False -> always use preset parameters for this model.
#
# Ridge is the only active model (per the fold-4 fix work); the other
# families remain implemented and can be re-enabled by flipping include.
MODEL_PIPELINE_CONFIG = {
    "Ridge": {
        "include": True,
        "tune": False,
    },
    "Lasso": {
        "include": False,
        "tune": False,
    },
    "Ridge_w": {
        "include": False,
        "tune": False,
    },
    "Lasso_w": {
        "include": False,
        "tune": False,
    },
    "Huber": {
        "include": False,
        "tune": False,
    },
}

_STD_FLOOR = 0.1
_NORM_FLOOR = 0.0001

_TARGET_COL = "returns"

_ALL_MODEL_NAMES = [
    "Ridge",
    "Lasso",
    "Ridge_w",
    "Lasso_w",
    "Huber",
]

_LINEAR_MODEL_NAMES = list(_ALL_MODEL_NAMES)

# Fold-4 fix: alpha raised from 50 to 9600. With ~200k standardized training
# rows the ridge penalty at this level mainly shrinks the ill-conditioned
# (low-variance) direction spanned by the collinear pair
# yret_price_dev_60 / ret_factor_zscore_60_yw_2_55, whose fitted contrast was
# unstable and flipped sign in the fold-4 validation window.
_RIDGE_ALPHA = 9600.0
_LASSO_ALPHA = 0.0001
_WINSOR_THRESH = 0.15

# Fold-4 fix: after standardization, features are clipped to +/- this many
# standard deviations before fitting and predicting. Caps the leverage of
# heavy-tailed feature outliers and stabilises the fit across regimes.
_FEATURE_CLIP_Z = 2.5

_HUBER_EPS = 1.35
_HUBER_A = 0.0001

_CV_TRAIN_PCTS = [0.50, 0.60, 0.70, 0.80, 0.90]
_CV_VAL_PCT = 0.10

# Grid re-centred around the selected preset (used only if tuning is enabled).
_RIDGE_ALPHA_GRID = [50.0, 3200.0, 6400.0, 9600.0, 12800.0, 19200.0]
_LASSO_ALPHA_GRID = [1e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
_WINSOR_THRESH_GRID = [0.05, 0.10, 0.15, 0.20, 0.25]
_HUBER_EPS_GRID = np.round(np.arange(1.05, 1.35 + 1e-12, 0.05), 2)

_N_JOBS = -1

FEATURE_COLS = [
    # ========================================================
    # Return-only feature set selected by five-fold CV.
    # Every feature below is computed only from xret/yret and
    # session-safe transformations of those return series.
    # None of the previous price/cross-price feature columns is used.
    #
    # Fold-4 fix: "xret_price_dev_6" was removed from this list. Its
    # validation correlation with the target decayed to ~0.015 in the
    # fold-4 window and subset searches at multiple alphas never placed
    # it in any of the top all-fold-positive subsets. The column is still
    # computed in _add_return_features for inspection/reuse.
    # ========================================================
    "ret_spread_ewma_prodpair_360_60",
    "yret_price_dev_60",
    "yret_price_dev_360_rsi_360",
    "ret_factor_zscore_60_yw_2_55",
    "yret_sum_accel_60_180",
    "yret_price_dev_120_rsi_60",
    "ret_ewm_spread_60",
]


# ============================================================
# Model configuration utilities
# ============================================================
def _active_model_names() -> list[str]:
    active = [
        nm for nm in _ALL_MODEL_NAMES
        if bool(MODEL_PIPELINE_CONFIG.get(nm, {}).get("include", False))
    ]
    if not active:
        raise ValueError(
            "No models are enabled. Set at least one MODEL_PIPELINE_CONFIG[model]['include'] to True."
        )
    return active


def _model_tuning_enabled(model_name: str) -> bool:
    return bool(
        ENABLE_HYPERPARAMETER_TUNING
        and MODEL_PIPELINE_CONFIG.get(model_name, {}).get("tune", False)
    )


# ============================================================
# Small utilities
# ============================================================
def _fmt_seconds(seconds: float) -> str:
    return f"{seconds:.4f} sec"


def _fmt_float(x: float, nd: int = 6) -> str:
    return "nan" if pd.isna(x) else f"{x:.{nd}f}"


def _fmt_pct(x: float) -> str:
    return "nan" if pd.isna(x) else f"{x:+.2f}%"


def _mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    return float(np.mean((y_true - y_pred) ** 2))


def _gain(model_mspe: float, baseline_mspe: float) -> float:
    if baseline_mspe == 0 or pd.isna(baseline_mspe):
        return float("nan")
    return 100.0 * (baseline_mspe - model_mspe) / baseline_mspe


def _r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) < 2 or len(y_pred) < 2:
        return float("nan")
    if np.allclose(np.std(y_true), 0) or np.allclose(np.std(y_pred), 0):
        return 0.0
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return float(corr ** 2) if not np.isnan(corr) else float("nan")


def _as_scalar_intercept(val: Any) -> float:
    arr = np.asarray(val, float)
    return 0.0 if arr.size == 0 else float(arr.ravel()[0])


def _round_half_up(x: float) -> float:
    return float(np.floor(x + 0.5))


def _parallel_map(func, items: list, n_jobs: int = _N_JOBS) -> list:
    if Parallel is None or delayed is None or len(items) <= 1:
        return [func(x) for x in items]
    return Parallel(n_jobs=n_jobs, prefer="processes")(delayed(func)(x) for x in items)


# ============================================================
# Price adjustments
# ============================================================
def _compute_price_adjustments(train_df: pd.DataFrame) -> dict[str, float]:
    x_min = train_df["xprice"].astype(float).min()
    y_min = train_df["yprice"].astype(float).min()
    x_adjust = _round_half_up(x_min - 1.0)
    y_adjust = _round_half_up(y_min - 1.0)

    print("=" * 120)
    print("DYNAMIC PRICE ADJUSTMENTS FROM TRAIN SET")
    print("=" * 120)
    print(f"xprice minimum                : {x_min}")
    print(f"xprice minimum minus 1        : {x_min - 1.0}")
    print(f"xprice rounded adjustment used: {x_adjust}")
    print()
    print(f"yprice minimum                : {y_min}")
    print(f"yprice minimum minus 1        : {y_min - 1.0}")
    print(f"yprice rounded adjustment used: {y_adjust}")
    print("=" * 120)
    print()

    return {"xprice_adjust": x_adjust, "yprice_adjust": y_adjust}


# ============================================================
# Feature engineering helpers
# ============================================================
def _daily_ewma_by_day(series: pd.Series, day_key: pd.Series, halflife: int) -> pd.Series:
    out = pd.Series(index=series.index, dtype=float)
    for d in day_key.dropna().unique():
        mask = day_key == d
        out.loc[mask] = series.loc[mask].ewm(halflife=halflife).mean().to_numpy()
    return out


def _daily_ewma_by_day_adjust_false(
    series: pd.Series,
    day_key: pd.Series,
    halflife: int,
) -> pd.Series:
    """Session-reset EWM using the recursive adjust=False definition."""
    out = pd.Series(index=series.index, dtype=float)
    safe = series.astype(float).fillna(0.0)
    for d in day_key.dropna().unique():
        mask = day_key == d
        out.loc[mask] = safe.loc[mask].ewm(
            halflife=halflife,
            adjust=False,
        ).mean().to_numpy()
    return out


def _rolling_by_session(
    series: pd.Series,
    day_key: pd.Series,
    window_seconds: int,
    agg: str,
    std_floor: float | None = None,
) -> pd.Series:
    out = pd.Series(index=series.index, dtype=float)
    for d in day_key.dropna().unique():
        mask = day_key == d
        s = series.loc[mask]
        if agg == "mean":
            vals = s.rolling(window=f"{window_seconds}s", closed="right").mean()
        elif agg == "sum":
            vals = s.rolling(window=f"{window_seconds}s", closed="right").sum()
        elif agg == "std":
            vals = s.rolling(window=f"{window_seconds}s", closed="right").std(ddof=0)
            vals = vals.fillna(0.0)
            if std_floor is not None:
                vals = vals + std_floor
        else:
            raise ValueError(f"Unsupported agg: {agg}")
        out.loc[mask] = vals.to_numpy()
    return out


def _lag_by_session(series: pd.Series, day_key: pd.Series, lag: int) -> pd.Series:
    return series.groupby(day_key).shift(lag)


def _rsi_series(px: np.ndarray, period: int = 14) -> np.ndarray:
    """Return a strictly causal RSI series.

    At row ``i``, the result uses only ``px[:i + 1]``:

    * During the first ``period`` observed changes, average gain and average
      loss are expanding means over the changes available so far.
    * After the warm-up, the averages follow Wilder's recursive smoothing.

    The first row has no preceding change and is assigned the neutral RSI
    value 50. Flat histories are also treated as neutral.
    """
    values = np.asarray(px, dtype=float)
    n = values.size

    if period <= 0:
        raise ValueError("RSI period must be a positive integer.")
    if n == 0:
        return np.asarray([], dtype=float)

    rsi = np.full(n, 50.0, dtype=float)
    avg_gain = 0.0
    avg_loss = 0.0

    for i in range(1, n):
        current = values[i]
        previous = values[i - 1]

        # A missing current/previous value contains no usable new information.
        # Keep the previous RSI and do not update the state. In this script the
        # selected RSI inputs are normally finite, but this makes the helper
        # robust when called independently.
        if not np.isfinite(current) or not np.isfinite(previous):
            rsi[i] = rsi[i - 1]
            continue

        delta = current - previous
        gain = max(delta, 0.0)
        loss = max(-delta, 0.0)

        if i <= period:
            # Strictly causal warm-up: expanding means over changes 1..i.
            avg_gain = ((i - 1) * avg_gain + gain) / i
            avg_loss = ((i - 1) * avg_loss + loss) / i
        else:
            # Wilder smoothing after the first ``period`` changes.
            avg_gain = ((period - 1) * avg_gain + gain) / period
            avg_loss = ((period - 1) * avg_loss + loss) / period

        if avg_loss == 0.0:
            rsi[i] = 50.0 if avg_gain == 0.0 else 100.0
        elif avg_gain == 0.0:
            rsi[i] = 0.0
        else:
            rs = avg_gain / avg_loss
            rsi[i] = 100.0 - 100.0 / (1.0 + rs)

    return rsi


def _rsi_by_session(series: pd.Series, day_key: pd.Series, period: int) -> pd.Series:
    out = pd.Series(index=series.index, dtype=float)
    for d in day_key.dropna().unique():
        mask = day_key == d
        vals = series.loc[mask].astype(float).to_numpy()
        out.loc[mask] = _rsi_series(vals, period=period)
    return out


def _add_return_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add the CV-selected feature set using only xret/yret information."""
    out = df.copy()
    if "datetime" not in out.columns:
        raise ValueError("Expected 'datetime' column before return feature engineering.")

    out["datetime"] = pd.to_datetime(out["datetime"])
    out = out.sort_values("datetime").reset_index(drop=True)

    original_index = out.index
    out = out.set_index("datetime", drop=False)
    day_key = out["datetime"].dt.normalize()

    xret_safe = out["xret"].astype(float).fillna(0.0)
    yret_safe = out["yret"].astype(float).fillna(0.0)
    xret_path = xret_safe.groupby(day_key).cumsum()
    yret_path = yret_safe.groupby(day_key).cumsum()

    # Still computed for inspection/reuse; no longer part of FEATURE_COLS
    # (fold-4 fix: signal decayed in the fold-4 validation window).
    xret_path_mean_6 = _rolling_by_session(xret_path, day_key, 60, "mean")
    out["xret_price_dev_6"] = xret_path - xret_path_mean_6

    yret_path_mean_60 = _rolling_by_session(yret_path, day_key, 600, "mean")
    out["yret_price_dev_60"] = yret_path - yret_path_mean_60

    yret_path_mean_120 = _rolling_by_session(yret_path, day_key, 1200, "mean")
    yret_price_dev_120 = yret_path - yret_path_mean_120
    out["yret_price_dev_120_rsi_60"] = _rsi_by_session(
        yret_price_dev_120, day_key, 60
    )

    yret_path_mean_360 = _rolling_by_session(yret_path, day_key, 3600, "mean")
    yret_price_dev_360 = yret_path - yret_path_mean_360
    out["yret_price_dev_360_rsi_360"] = _rsi_by_session(
        yret_price_dev_360, day_key, 360
    )

    ret_spread_path = yret_path - xret_path
    ret_spread_ewm_60 = _daily_ewma_by_day(ret_spread_path, day_key, 60)
    ret_spread_ewm_360 = _daily_ewma_by_day(ret_spread_path, day_key, 360)
    out["ret_spread_ewma_prodpair_360_60"] = (
        (ret_spread_path - ret_spread_ewm_360)
        * (ret_spread_path - ret_spread_ewm_60)
    )

    ret_factor_path = xret_path + 2.55 * yret_path
    ret_factor_mean_60 = _rolling_by_session(ret_factor_path, day_key, 600, "mean")
    ret_factor_std_60 = _rolling_by_session(
        ret_factor_path, day_key, 600, "std", _STD_FLOOR
    )
    out["ret_factor_zscore_60_yw_2_55"] = (
        (ret_factor_path - ret_factor_mean_60) / ret_factor_std_60
    )

    yret_sum_60 = _rolling_by_session(out["yret"].astype(float), day_key, 600, "sum")
    yret_sum_180 = _rolling_by_session(out["yret"].astype(float), day_key, 1800, "sum")
    out["yret_sum_accel_60_180"] = yret_sum_60 - (60.0 / 180.0) * yret_sum_180

    xret_ewm_60 = _daily_ewma_by_day_adjust_false(out["xret"], day_key, 60)
    yret_ewm_60 = _daily_ewma_by_day_adjust_false(out["yret"], day_key, 60)
    out["ret_ewm_spread_60"] = yret_ewm_60 - xret_ewm_60

    out.index = original_index
    return out


def _add_cross_features(df: pd.DataFrame, price_adjustments: dict[str, float]) -> pd.DataFrame:
    out = df.copy()
    if "datetime" not in out.columns:
        raise ValueError("Expected 'datetime' column before feature engineering.")

    out["datetime"] = pd.to_datetime(out["datetime"])
    out = out.sort_values("datetime").reset_index(drop=True)

    x_adj = out["xprice"].astype(float) - price_adjustments["xprice_adjust"]
    y_adj = out["yprice"].astype(float) - price_adjustments["yprice_adjust"]

    prod_xy = x_adj * y_adj
    safe_prod_xy = np.where(prod_xy >= 0, prod_xy, np.nan)
    safe_x = x_adj.replace(0, np.nan)
    safe_y = y_adj.replace(0, np.nan)

    out["x_adj"] = x_adj
    out["y_adj"] = y_adj
    out["yx_spread_raw"] = y_adj - x_adj
    out["xy_relation_raw"] = safe_x / safe_y
    out["xy_square_raw"] = np.sqrt(x_adj ** 2 + y_adj ** 2) / 2.0
    out["xy_geom_raw"] = np.sqrt(safe_prod_xy)
    out["xlog_raw"] = np.log1p(x_adj)
    out["ylog_raw"] = np.log1p(y_adj)
    out["day_norm"] = out["datetime"].dt.normalize()

    original_index = out.index
    out = out.set_index("datetime", drop=False)

    xy_geom_roll_mean_6 = _rolling_by_session(out["xy_geom_raw"], out["day_norm"], 60, "mean")
    out["xy_geom_time_mean_6"] = out["xy_geom_raw"] - xy_geom_roll_mean_6

    xy_geom_roll_mean_60 = _rolling_by_session(out["xy_geom_raw"], out["day_norm"], 600, "mean")
    out["xy_geom_time_mean_60"] = out["xy_geom_raw"] - xy_geom_roll_mean_60
    out["xy_geom_time_mean_60_daily_ewma_6"] = _daily_ewma_by_day(out["xy_geom_time_mean_60"], out["day_norm"], 6)

    out["xy_relation_time_std_360"] = _rolling_by_session(
        out["xy_relation_raw"], out["day_norm"], 3600, "std", _STD_FLOOR
    )

    xy_square_mean_60 = _rolling_by_session(out["xy_square_raw"], out["day_norm"], 600, "mean")
    xy_square_std_60 = _rolling_by_session(out["xy_square_raw"], out["day_norm"], 600, "std", _STD_FLOOR)
    out["xy_square_time_zscore_60"] = (out["xy_square_raw"] - xy_square_mean_60) / xy_square_std_60

    yx_spread_daily_ewma_60 = _daily_ewma_by_day(out["yx_spread_raw"], out["day_norm"], 60)
    yx_spread_daily_ewma_360 = _daily_ewma_by_day(out["yx_spread_raw"], out["day_norm"], 360)
    out["yx_spread_ewma_prodpair_360_60"] = (
        (out["yx_spread_raw"] - yx_spread_daily_ewma_360)
        * (out["yx_spread_raw"] - yx_spread_daily_ewma_60)
    )

    xprice_roll_mean_6 = _rolling_by_session(out["x_adj"], out["day_norm"], 60, "mean")
    out["xprice_time_mean_6"] = out["x_adj"] - xprice_roll_mean_6

    xprice_roll_mean_60 = _rolling_by_session(out["x_adj"], out["day_norm"], 600, "mean")
    out["xprice_time_mean_60"] = out["x_adj"] - xprice_roll_mean_60

    xprice_daily_ewma_24_raw = _daily_ewma_by_day(out["x_adj"], out["day_norm"], 24)
    out["xprice_daily_ewma_24"] = out["x_adj"] - xprice_daily_ewma_24_raw

    out["xprice_time_mean_60_daily_ewma_60"] = _daily_ewma_by_day(
        out["xprice_time_mean_60"], out["day_norm"], 60
    )

    xlog_daily_ewma_60_raw = _daily_ewma_by_day(out["xlog_raw"], out["day_norm"], 60)
    out["xlog_daily_ewma_60"] = out["xlog_raw"] - xlog_daily_ewma_60_raw

    out["xprice_time_mean_6_rsi_6"] = _rsi_by_session(out["xprice_time_mean_6"], out["day_norm"], 6)

    yprice_roll_mean_60 = _rolling_by_session(out["y_adj"], out["day_norm"], 600, "mean")
    out["yprice_time_mean_60"] = out["y_adj"] - yprice_roll_mean_60

    yprice_roll_mean_360 = _rolling_by_session(out["y_adj"], out["day_norm"], 3600, "mean")
    out["yprice_time_mean_360"] = out["y_adj"] - yprice_roll_mean_360

    out["yprice_time_mean_60_lag_60"] = _lag_by_session(out["yprice_time_mean_60"], out["day_norm"], 60)

    yprice_std_360 = _rolling_by_session(out["y_adj"], out["day_norm"], 3600, "std", _STD_FLOOR)
    out["yprice_time_zscore_360"] = out["yprice_time_mean_360"] / yprice_std_360

    yprice_daily_ewma_24_raw = _daily_ewma_by_day(out["y_adj"], out["day_norm"], 24)
    yprice_daily_ewma_60_raw = _daily_ewma_by_day(out["y_adj"], out["day_norm"], 60)
    out["yprice_daily_ewma_24"] = out["y_adj"] - yprice_daily_ewma_24_raw
    out["yprice_daily_ewma_60"] = out["y_adj"] - yprice_daily_ewma_60_raw
    out["yprice_ewma_difpair_60_24"] = out["yprice_daily_ewma_60"] - out["yprice_daily_ewma_24"]

    out["yprice_time_mean_60_daily_ewma_24"] = _daily_ewma_by_day(
        out["yprice_time_mean_60"], out["day_norm"], 24
    )

    ylog_daily_ewma_360_raw = _daily_ewma_by_day(out["ylog_raw"], out["day_norm"], 360)
    out["ylog_daily_ewma_360"] = out["ylog_raw"] - ylog_daily_ewma_360_raw

    out["yprice_time_mean_60_rsi_360"] = _rsi_by_session(out["yprice_time_mean_60"], out["day_norm"], 360)

    drop_cols = [
        "x_adj", "y_adj", "xlog_raw", "ylog_raw", "yx_spread_raw", "xy_relation_raw",
        "xy_square_raw", "xy_geom_raw", "xy_geom_time_mean_60", "xprice_time_mean_60",
        "yprice_daily_ewma_24", "day_norm",
    ]
    out = out.drop(columns=drop_cols, errors="ignore")
    out.index = original_index
    return out


def _build_frame(
    path: str,
    price_adjustments: Optional[dict[str, float]] = None,
) -> tuple[pd.DataFrame, dict[str, float]]:
    df = pd.read_csv(path)
    required = ["timestamp", "xprice", "yprice"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available columns: {list(df.columns)}")

    if price_adjustments is None:
        price_adjustments = _compute_price_adjustments(df)

    df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.sort_values("timestamp").reset_index(drop=True)

    cols = df.columns.tolist()
    ts_idx = cols.index("timestamp")
    if "datetime" in cols:
        cols.remove("datetime")
    cols.insert(ts_idx + 1, "datetime")
    df = df[cols]

    df["xret"] = df["xprice"].diff()
    df["yret"] = df["yprice"].diff()
    df = _add_cross_features(df, price_adjustments)
    df = _add_return_features(df)

    df["datetime"] = pd.to_datetime(df["datetime"])
    df["date"] = df["datetime"].dt.date

    lags = [1, 61, 121, 141, 181, 241, 301, 361]
    for lag in lags:
        df[f"xreturn_lag_{lag}"] = df.groupby("date")["xret"].shift(lag)
        df[f"yreturn_lag_{lag}"] = df.groupby("date")["yret"].shift(lag)

    open_cutoff_min = 10
    close_cutoff_min = 79
    _date = df["datetime"].dt.normalize()
    _session_open = df.groupby(_date)["datetime"].transform("min")
    _minute_from_open = (df["datetime"] - _session_open).dt.total_seconds() / 60.0

    df["day_opening"] = (_minute_from_open < open_cutoff_min).astype(int)
    df["midday"] = ((_minute_from_open >= open_cutoff_min) & (_minute_from_open < close_cutoff_min)).astype(int)
    df["day_closing"] = (_minute_from_open >= close_cutoff_min).astype(int)

    df["day"] = df["datetime"].dt.normalize()
    df["is_last_row_of_day"] = (
        df.groupby("day").cumcount() == df.groupby("day")["day"].transform("size") - 1
    )
    return df, price_adjustments


# ============================================================
# Scaling
# ============================================================
def _fit_feature_scaler(X_train_raw: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    means = X_train_raw.mean()
    stds = X_train_raw.std(ddof=0)
    return means, stds


def _apply_feature_scaler(
    X_raw: pd.DataFrame,
    means: pd.Series,
    stds: pd.Series,
    fill_values: pd.Series | None = None,
    clip_z: float | None = _FEATURE_CLIP_Z,
) -> pd.DataFrame:
    """Standardize features and clip them to +/- clip_z standard deviations.

    Fold-4 fix: clipping happens right after standardization (before NaN
    filling) so extreme feature outliers cannot dominate the linear fit.
    The same transform is applied identically at train, CV, and test time.
    """
    X_scaled = (X_raw - means) / (stds + _NORM_FLOOR)
    if clip_z is not None:
        X_scaled = X_scaled.clip(lower=-float(clip_z), upper=float(clip_z))
    if fill_values is not None:
        X_scaled = X_scaled.fillna(fill_values)
    return X_scaled


# ============================================================
# Model helpers
# ============================================================
def _winsorize_fixed(y: np.ndarray, tau: float) -> np.ndarray:
    return np.clip(np.asarray(y, float), -float(tau), float(tau))


def _winsorize_quantile(y: np.ndarray, q: float) -> np.ndarray:
    y = np.asarray(y, float)
    q = min(max(float(q), 0.0), 0.499999)
    if q == 0.0 or y.size == 0:
        return y.copy()
    lo = float(np.nanquantile(y, q))
    hi = float(np.nanquantile(y, 1.0 - q))
    return np.clip(y, lo, hi)


def _make_model(model_name: str, config: dict) -> Any:
    if model_name in ("Ridge", "Ridge_w"):
        return Ridge(alpha=float(config["alpha"]))

    if model_name in ("Lasso", "Lasso_w"):
        return Lasso(alpha=float(config["alpha"]), max_iter=20000)

    if model_name == "Huber":
        return HuberRegressor(
            epsilon=float(config.get("epsilon", _HUBER_EPS)),
            alpha=float(config.get("alpha", _HUBER_A)),
            max_iter=20000,
        )

    raise ValueError(f"Unknown model: {model_name}")


def _fit_one_model(model_name: str, X: np.ndarray, y: np.ndarray, config: dict) -> Any:
    model = _make_model(model_name, config)

    if model_name.endswith("_w"):
        if config.get("winsor_style", "fixed") == "quantile":
            y_fit = _winsorize_quantile(y, config["winsor_thresh"])
        else:
            y_fit = _winsorize_fixed(y, config["winsor_thresh"])
    else:
        y_fit = y

    model.fit(X, y_fit)
    return model


def _linear_predict_from_params(params: dict, model_name: str, X: np.ndarray, close_mask: np.ndarray) -> np.ndarray:
    p = X.dot(np.asarray(params[f"{model_name}_coef"], float)) + float(params[f"{model_name}_intercept"])
    p = np.asarray(p, float).ravel()
    p[close_mask] = 0.0
    return p


def _predict_fitted(
    fitted: dict[str, Any],
    X_scaled: np.ndarray,
    close_mask: np.ndarray,
    model_names: list[str],
) -> dict[str, np.ndarray]:
    preds = {}
    for nm in model_names:
        p = np.asarray(fitted[nm].predict(X_scaled), float).ravel()
        p[close_mask] = 0.0
        preds[nm] = p
    return preds


# ============================================================
# CV cache and tuning
# ============================================================
def _build_cv_cache(work: pd.DataFrame) -> list[dict]:
    unique_days = pd.Index(work["day"].drop_duplicates().sort_values())
    n_days = len(unique_days)
    if n_days < 20:
        raise ValueError(f"Not enough unique days for 5 rolling folds. Found only {n_days} days.")

    val_days_count = max(int(round(n_days * _CV_VAL_PCT)), 1)
    fold_cache = []

    for fold_num, train_pct in enumerate(_CV_TRAIN_PCTS, start=1):
        train_day_count = min(int(round(n_days * train_pct)), n_days - 1)
        val_start_idx = train_day_count
        val_end_idx = min(val_start_idx + val_days_count, n_days)
        if val_start_idx >= n_days or val_end_idx <= val_start_idx:
            continue

        train_days = unique_days[:train_day_count]
        val_days = unique_days[val_start_idx:val_end_idx]
        train_df = work[work["day"].isin(train_days)].copy()
        val_df = work[work["day"].isin(val_days)].copy()

        train_clean = train_df.dropna(subset=FEATURE_COLS + [_TARGET_COL]).copy()
        if train_clean.empty:
            continue

        val_score_mask = val_df[_TARGET_COL].notna().to_numpy()
        y_val_true = val_df.loc[val_score_mask, _TARGET_COL].to_numpy(dtype=float)
        if len(y_val_true) == 0:
            continue

        train_mean_y = float(train_clean[_TARGET_COL].mean())
        baseline_val_mspe = _mse(y_val_true, np.full(len(y_val_true), train_mean_y))

        X_train_raw_df = train_clean[FEATURE_COLS].copy()

        means, stds = _fit_feature_scaler(X_train_raw_df)
        X_train_scaled_df = _apply_feature_scaler(X_train_raw_df, means, stds)
        scaled_fill_values = X_train_scaled_df.mean()
        X_train_scaled_df = X_train_scaled_df.fillna(scaled_fill_values)
        X_val_scaled_df = _apply_feature_scaler(
            val_df[FEATURE_COLS].copy(),
            means,
            stds,
            fill_values=scaled_fill_values,
        )

        fold_cache.append({
            "fold": fold_num,
            "train_pct": train_pct,
            "train_days": len(train_days),
            "val_days": len(val_days),
            "baseline_val_mspe": baseline_val_mspe,

            "X_train": X_train_scaled_df.to_numpy(dtype=float),
            "X_val": X_val_scaled_df.to_numpy(dtype=float),

            "y_train": train_clean[_TARGET_COL].to_numpy(dtype=float),
            "close_train": train_clean["is_last_row_of_day"].to_numpy(dtype=bool),

            "y_val_full": val_df[_TARGET_COL].to_numpy(dtype=float),
            "val_score_mask": val_score_mask,
            "close_val": val_df["is_last_row_of_day"].to_numpy(dtype=bool),
        })

    if not fold_cache:
        raise RuntimeError("No valid CV folds were produced.")
    return fold_cache


def _preset_configs() -> dict:
    return {
        "Ridge": {"alpha": _RIDGE_ALPHA, "source": "preset"},
        "Lasso": {"alpha": _LASSO_ALPHA, "source": "preset"},
        "Ridge_w": {
            "alpha": _RIDGE_ALPHA,
            "winsor_thresh": _WINSOR_THRESH,
            "winsor_style": "fixed",
            "source": "preset",
        },
        "Lasso_w": {
            "alpha": _LASSO_ALPHA,
            "winsor_thresh": _WINSOR_THRESH,
            "winsor_style": "fixed",
            "source": "preset",
        },
        "Huber": {
            "epsilon": _HUBER_EPS,
            "alpha": _HUBER_A,
            "source": "preset",
        },
    }


def _eval_model_config_on_fold(args: tuple[str, dict, dict]) -> dict:
    model_name, config, fd = args

    fitted = _fit_one_model(model_name, fd["X_train"], fd["y_train"], config)
    pred = np.asarray(fitted.predict(fd["X_val"]), float).ravel()
    pred[fd["close_val"]] = 0.0

    mask = fd["val_score_mask"]
    mspe = _mse(fd["y_val_full"][mask], pred[mask])
    return {
        "fold": fd["fold"],
        "mspe": mspe,
        "gain": _gain(mspe, fd["baseline_val_mspe"]),
    }


def _evaluate_model_config(model_name: str, config: dict, cv_cache: list[dict]) -> dict:
    items = [(model_name, config, fd) for fd in cv_cache]
    fold_rows = _parallel_map(_eval_model_config_on_fold, items)
    gains = np.asarray([r["gain"] for r in fold_rows], float)
    mspes = np.asarray([r["mspe"] for r in fold_rows], float)
    return {
        "model": model_name,
        "config": dict(config),
        "fold_rows": fold_rows,
        "avg_gain": float(np.nanmean(gains)),
        "min_gain": float(np.nanmin(gains)),
        "std_gain": float(np.nanstd(gains)),
        "avg_mspe": float(np.nanmean(mspes)),
        "n_beat_baseline": int(np.sum(gains > 0)),
    }


def _select_best_eval(rows: list[dict]) -> dict:
    return sorted(
        rows,
        key=lambda r: (-r["avg_gain"], -r["min_gain"], r["std_gain"], r["avg_mspe"]),
    )[0]


def _tuned_candidates(model_name: str) -> list[dict]:
    if model_name == "Ridge":
        return [{"alpha": a, "source": "tuned"} for a in _RIDGE_ALPHA_GRID]

    if model_name == "Lasso":
        return [{"alpha": a, "source": "tuned"} for a in _LASSO_ALPHA_GRID]

    if model_name == "Ridge_w":
        return [
            {
                "alpha": a,
                "winsor_thresh": w,
                "winsor_style": "quantile",
                "source": "tuned",
            }
            for a in _RIDGE_ALPHA_GRID
            for w in _WINSOR_THRESH_GRID
        ]

    if model_name == "Lasso_w":
        return [
            {
                "alpha": a,
                "winsor_thresh": w,
                "winsor_style": "quantile",
                "source": "tuned",
            }
            for a in _LASSO_ALPHA_GRID
            for w in _WINSOR_THRESH_GRID
        ]

    if model_name == "Huber":
        return [
            {"epsilon": float(eps), "alpha": _HUBER_A, "source": "tuned"}
            for eps in _HUBER_EPS_GRID
        ]

    return []


def _evaluate_candidate_wrapper(args: tuple[str, dict, list[dict]]) -> dict:
    model_name, cfg, cv_cache = args
    return _evaluate_model_config(model_name, cfg, cv_cache)


def _run_hyperparameter_tuning(cv_cache: list[dict], model_names: list[str]) -> dict:
    if not ENABLE_HYPERPARAMETER_TUNING:
        print("----->Hyperparameter tuning globally disabled. Using preset configurations only.")

    base_all = _preset_configs()
    selected = {}
    switch_notes = []
    search_results = {}

    if ENABLE_HYPERPARAMETER_TUNING:
        print("----->Running preset-vs-tuned hyperparameter search...")
        print("       A model is tuned only if its MODEL_PIPELINE_CONFIG entry has tune=True.")

    for nm in model_names:
        base_cfg = dict(base_all[nm])
        preset_eval = _evaluate_model_config(nm, base_cfg, cv_cache)

        should_tune = _model_tuning_enabled(nm)
        candidates = _tuned_candidates(nm) if should_tune else []

        if not candidates:
            selected[nm] = base_cfg
            selected[nm]["source"] = "preset"

            if ENABLE_HYPERPARAMETER_TUNING and not MODEL_PIPELINE_CONFIG.get(nm, {}).get("tune", False):
                higher = "Preset - tuning disabled for model"
            elif not ENABLE_HYPERPARAMETER_TUNING:
                higher = "Preset - global tuning disabled"
            else:
                higher = "Preset"

            switch_notes.append({
                "family": nm,
                "included": True,
                "tuning_allowed_for_model": bool(MODEL_PIPELINE_CONFIG.get(nm, {}).get("tune", False)),
                "switched_to_tuned": False,
                "preset_mean_improv_pct": preset_eval["avg_gain"],
                "tuned_mean_improv_pct": float("nan"),
                "higher": higher,
                "preset_config": preset_eval["config"],
                "tuned_config": None,
                "selected_config": selected[nm],
            })
            search_results[nm] = {
                "preset": preset_eval,
                "tuned_best": None,
                "all_tuned": [],
            }
            continue

        print(f"       Tuning {nm}...")
        candidate_items = [(nm, cfg, cv_cache) for cfg in candidates]
        tuned_rows = _parallel_map(_evaluate_candidate_wrapper, candidate_items)
        tuned_best = _select_best_eval(tuned_rows)

        switched = bool(tuned_best["avg_gain"] > preset_eval["avg_gain"])
        selected[nm] = dict(tuned_best["config"] if switched else base_cfg)
        selected[nm]["source"] = "tuned" if switched else "preset"

        switch_notes.append({
            "family": nm,
            "included": True,
            "tuning_allowed_for_model": True,
            "switched_to_tuned": switched,
            "preset_mean_improv_pct": preset_eval["avg_gain"],
            "tuned_mean_improv_pct": tuned_best["avg_gain"],
            "higher": "Tuned" if tuned_best["avg_gain"] > preset_eval["avg_gain"] else "Preset/Equal",
            "preset_config": preset_eval["config"],
            "tuned_config": tuned_best["config"],
            "selected_config": selected[nm],
        })
        search_results[nm] = {
            "preset": preset_eval,
            "tuned_best": tuned_best,
            "all_tuned": tuned_rows,
        }

    _print_switch_notes(switch_notes)
    return {
        "enabled": bool(ENABLE_HYPERPARAMETER_TUNING),
        "model_pipeline_config": dict(MODEL_PIPELINE_CONFIG),
        "active_model_names": list(model_names),
        "selected_configs": selected,
        "switch_notes": switch_notes,
        "search_results": search_results,
    }


def _print_switch_notes(notes: list[dict]) -> None:
    bar = "=" * 130
    print(f"\n{bar}")
    print("HYPERPARAMETER SWITCH DECISIONS")
    print("Switch to tuned only if tuned mean validation improvement beats preset mean validation improvement.")
    print("Models with include=False are excluded before this stage.")
    print(bar)
    hdr = (
        f"{'Family':<16}"
        f"{'Tune Allowed?':>15}"
        f"{'Switched?':>12}"
        f"{'Preset Mean Δ%':>18}"
        f"{'Tuned Mean Δ%':>18}"
        f"{'Higher':>34}"
    )
    print(hdr)
    print("-" * len(hdr))
    for n in notes:
        print(
            f"{n['family']:<16}"
            f"{str(n['tuning_allowed_for_model']):>15}"
            f"{str(n['switched_to_tuned']):>12}"
            f"{_fmt_pct(n['preset_mean_improv_pct']):>18}"
            f"{_fmt_pct(n['tuned_mean_improv_pct']):>18}"
            f"{n['higher']:>34}"
        )
        print(f"    preset : {n['preset_config']}")
        print(f"    tuned  : {n['tuned_config']}")
        print(f"    chosen : {n['selected_config']}")
    print(bar + "\n")


# ============================================================
# CV reporting and final fitting
# ============================================================
def _run_one_cv_fold(args: tuple[dict, dict, list[str]]) -> dict:
    fd, configs, model_names = args

    fitted = {}
    for nm in model_names:
        fitted[nm] = _fit_one_model(nm, fd["X_train"], fd["y_train"], configs[nm])

    preds_train = _predict_fitted(
        fitted,
        X_scaled=fd["X_train"],
        close_mask=fd["close_train"],
        model_names=model_names,
    )
    preds_val = _predict_fitted(
        fitted,
        X_scaled=fd["X_val"],
        close_mask=fd["close_val"],
        model_names=model_names,
    )

    rec = {
        "fold": fd["fold"],
        "train_pct": fd["train_pct"],
        "train_days": fd["train_days"],
        "val_days": fd["val_days"],
        "baseline_val_mspe": fd["baseline_val_mspe"],
    }

    mask = fd["val_score_mask"]
    for nm in model_names:
        train_pred = preds_train[nm]
        val_pred = preds_val[nm][mask]
        y_val = fd["y_val_full"][mask]
        rec[f"{nm}_train_r2"] = _r2(fd["y_train"], train_pred)
        rec[f"{nm}_train_mspe"] = _mse(fd["y_train"], train_pred)
        rec[f"{nm}_val_r2"] = _r2(y_val, val_pred)
        rec[f"{nm}_val_mspe"] = _mse(y_val, val_pred)
        rec[f"{nm}_val_improv_pct"] = _gain(rec[f"{nm}_val_mspe"], fd["baseline_val_mspe"])

    return rec


def _run_cv(cv_cache: list[dict], tuning: dict, model_names: list[str]) -> list[dict]:
    selected = tuning["selected_configs"]
    bar = "=" * 120
    print(f"\n{bar}")
    print("SLIDING-WINDOW CROSS-VALIDATION  (5 folds, day-aligned boundaries)")
    print("Baseline: predict the training-fold mean return for every validation row.")
    print(f"Features: {FEATURE_COLS}")
    print(f"Feature clip (post-standardization): +/-{_FEATURE_CLIP_Z} std")
    print(f"Active models: {model_names}")
    print(f"Selected configs: {selected}")
    print(bar)

    history = _parallel_map(_run_one_cv_fold, [(fd, selected, model_names) for fd in cv_cache])
    history = sorted(history, key=lambda r: r["fold"])

    for rec in history:
        print(
            f"\nFold {rec['fold']} | train ≈ {int(round(rec['train_pct'] * 100))}% | "
            f"train days: {rec['train_days']} | val days: {rec['val_days']} | "
            f"baseline MSPE (val): {rec['baseline_val_mspe']:.8f}"
        )
        hdr = f"{'Model':<18}{'Val R²':>12}{'Val MSPE':>14}{'vs Baseline':>15}{'Train R²':>12}{'Train MSPE':>14}"
        print(hdr)
        print("-" * len(hdr))
        for nm in model_names:
            print(
                f"{nm:<18}"
                f"{_fmt_float(rec[f'{nm}_val_r2'], 6):>12}"
                f"{_fmt_float(rec[f'{nm}_val_mspe'], 8):>14}"
                f"{_fmt_pct(rec[f'{nm}_val_improv_pct']):>15}"
                f"{_fmt_float(rec[f'{nm}_train_r2'], 6):>12}"
                f"{_fmt_float(rec[f'{nm}_train_mspe'], 8):>14}"
            )

    _print_cv_averages(history, model_names)
    return history


def _print_cv_averages(history: list[dict], model_names: list[str]) -> None:
    bar = "=" * 120
    print(f"\n{bar}")
    print("CV AVERAGES - ACTIVE MODELS")
    hdr = f"{'Model':<18}{'Avg Val R²':>14}{'Avg Val MSPE':>16}{'Avg vs Baseline':>18}{'Avg Train R²':>15}{'Avg Train MSPE':>17}"
    print(hdr)
    print("-" * len(hdr))
    for nm in model_names:
        print(
            f"{nm:<18}"
            f"{_fmt_float(np.mean([r[f'{nm}_val_r2'] for r in history]), 6):>14}"
            f"{_fmt_float(np.mean([r[f'{nm}_val_mspe'] for r in history]), 8):>16}"
            f"{_fmt_pct(np.mean([r[f'{nm}_val_improv_pct'] for r in history])):>18}"
            f"{_fmt_float(np.mean([r[f'{nm}_train_r2'] for r in history]), 6):>15}"
            f"{_fmt_float(np.mean([r[f'{nm}_train_mspe'] for r in history]), 8):>17}"
        )
    print(bar + "\n")


def _summarize_model(nm: str, history: list[dict]) -> dict:
    improvs = [r[f"{nm}_val_improv_pct"] for r in history]
    val_mspes = [r[f"{nm}_val_mspe"] for r in history]
    train_mspes = [r[f"{nm}_train_mspe"] for r in history]
    val_r2s = [r[f"{nm}_val_r2"] for r in history]
    return {
        "name": nm,
        "mean_improv": float(np.nanmean(improvs)),
        "std_improv": float(np.nanstd(improvs)),
        "min_improv": float(np.nanmin(improvs)),
        "max_improv": float(np.nanmax(improvs)),
        "n_beat_baseline": int(np.sum(np.asarray(improvs) > 0)),
        "n_folds": len(improvs),
        "mean_val_mspe": float(np.nanmean(val_mspes)),
        "mean_val_r2": float(np.nanmean(val_r2s)),
        "mean_train_mspe": float(np.nanmean(train_mspes)),
    }


def _select_best_model(history: list[dict], model_names: list[str]) -> str:
    summaries = [_summarize_model(nm, history) for nm in model_names]
    summaries.sort(
        key=lambda s: (
            -s["mean_improv"],
            -s["min_improv"],
            s["std_improv"],
            s["mean_val_mspe"],
        )
    )
    bar = "=" * 120
    print(f"\n{bar}")
    print("BEST OVERALL MODEL SELECTION")
    print("Primary criterion : mean validation improvement vs baseline (%)")
    print("Tiebreakers       : (1) worst-fold Δ%, (2) std of Δ%, (3) mean val MSPE")
    print(bar)
    hdr = f"{'Rank':<6}{'Model':<18}{'Mean Δ%':>10}{'Std Δ%':>10}{'Min Δ%':>10}{'Max Δ%':>10}{'Beat':>8}{'Val MSPE':>14}{'Gap':>14}"
    print(hdr)
    print("-" * len(hdr))
    for rank, s in enumerate(summaries, 1):
        gap = s["mean_val_mspe"] - s["mean_train_mspe"]
        beat = f"{s['n_beat_baseline']}/{s['n_folds']}"
        print(
            f"{rank:<6}{s['name']:<18}"
            f"{_fmt_pct(s['mean_improv']):>10}{_fmt_pct(s['std_improv']):>10}"
            f"{_fmt_pct(s['min_improv']):>10}{_fmt_pct(s['max_improv']):>10}"
            f"{beat:>8}{_fmt_float(s['mean_val_mspe'], 8):>14}{_fmt_float(gap, 8):>14}"
        )
    print(f"\nBEST OVERALL MODEL: {summaries[0]['name']}")
    print(bar + "\n")
    return summaries[0]["name"]


def _print_feature_importances(coefs: np.ndarray, feature_names: list[str], label: str) -> None:
    abs_coefs = np.abs(np.asarray(coefs, float))
    total = float(abs_coefs.sum())
    print(f"------>Feature importances ({label}):")
    if total == 0:
        print("  (all coefficients/importances are zero)")
        return
    pairs = sorted(zip(feature_names, coefs, abs_coefs / total * 100), key=lambda x: -x[2])
    for feat, coef, pct in pairs:
        print(f"  {feat:<45} {pct:>6.2f}%   {coef:>10.6f}")
    print()


def _check_prediction_integrity(
    preds: dict[str, np.ndarray],
    close_mask: np.ndarray,
    model_names: list[str],
    label: str = "TEST",
) -> None:
    bar = "=" * 100
    print(f"\n{bar}")
    print(f"{label} PREDICTION INTEGRITY CHECKS")
    print(bar)
    n_rows = len(close_mask)
    n_close = int(np.sum(close_mask))
    hdr = f"{'Model':<18}{'Rows':>10}{'Any NaN?':>12}{'All rows predicted?':>22}{'Close rows zero?':>18}"
    print(hdr)
    print("-" * len(hdr))
    for nm in model_names:
        p = np.asarray(preds[nm], float).ravel()
        print(
            f"{nm:<18}{p.shape[0]:>10}{str(np.isnan(p).any()):>12}"
            f"{str(p.shape[0] == n_rows):>22}{str(np.allclose(p[close_mask], 0.0) if n_close else True):>18}"
        )
    print(f"\nTotal test rows          : {n_rows}")
    print(f"Closing rows zeroed     : {n_close}")
    print(bar + "\n")


# ============================================================
# Public API
# ============================================================
def modelEstimate(trainingFilename: str) -> dict:
    model_names = _active_model_names()

    train_start = perf_counter()
    print("--------->Start Estimation...")
    print(f"Active models: {model_names}")
    print(f"Model pipeline config: {MODEL_PIPELINE_CONFIG}")
    print(f"Global hyperparameter tuning enabled: {ENABLE_HYPERPARAMETER_TUNING}")

    work, price_adjustments = _build_frame(trainingFilename)
    if _TARGET_COL not in work.columns:
        raise ValueError(f"Training file must contain target column '{_TARGET_COL}'.")
    work = work.sort_values(["day", "datetime"]).reset_index(drop=True)

    print("----->Building cached CV folds...")
    cv_cache = _build_cv_cache(work)

    tuning = _run_hyperparameter_tuning(cv_cache, model_names)
    selected_configs = tuning["selected_configs"]

    history = _run_cv(cv_cache, tuning, model_names)
    best_model = _select_best_model(history, model_names)

    print("------>Normalization...")
    full_train = work.dropna(subset=FEATURE_COLS + [_TARGET_COL]).copy()

    X_raw_df = full_train[FEATURE_COLS].copy()
    raw_fill_values = X_raw_df.mean()
    X_raw_filled_df = X_raw_df.fillna(raw_fill_values)

    y = full_train[_TARGET_COL].to_numpy(dtype=float)

    means, stds = _fit_feature_scaler(X_raw_df)
    X_scaled_df = _apply_feature_scaler(X_raw_df, means, stds)
    scaled_fill_values = X_scaled_df.mean()
    X_scaled_df = X_scaled_df.fillna(scaled_fill_values)
    X_scaled = X_scaled_df.to_numpy(dtype=float)

    close_mask = full_train["is_last_row_of_day"].to_numpy(dtype=bool)

    baseline_mean = float(np.mean(y))
    baseline_mspe = _mse(y, np.full(len(y), baseline_mean))

    print("------>Fitting final selected models...")
    fitted = {}
    for nm in model_names:
        fitted[nm] = _fit_one_model(nm, X_scaled, y, selected_configs[nm])

    preds_train = _predict_fitted(
        fitted,
        X_scaled=X_scaled,
        close_mask=close_mask,
        model_names=model_names,
    )

    bar = "=" * 100
    print(f"\n{bar}")
    print("TRAIN-SET METRICS - ACTIVE MODELS")
    print(f"Baseline MSPE (training mean = {baseline_mean:.6f}): {baseline_mspe:.8f}")
    hdr = f"{'Model':<18}{'Train R²':>12}{'Train MSPE':>14}{'vs Baseline':>15}"
    print(hdr)
    print("-" * len(hdr))
    for nm in model_names:
        train_mspe = _mse(y, preds_train[nm])
        print(
            f"{nm:<18}"
            f"{_fmt_float(_r2(y, preds_train[nm]), 6):>12}"
            f"{_fmt_float(train_mspe, 8):>14}"
            f"{_fmt_pct(_gain(train_mspe, baseline_mspe)):>15}"
        )
    print(bar + "\n")

    for nm in model_names:
        _print_feature_importances(fitted[nm].coef_, FEATURE_COLS, nm)

    train_elapsed = perf_counter() - train_start
    print(f"Training time: {_fmt_seconds(train_elapsed)}")

    params: dict = {
        "trainingFilename": trainingFilename,
        "price_adjustments": price_adjustments,
        "feature_cols": FEATURE_COLS,

        "norm_mean": means,
        "norm_std": stds,
        "scaled_fill_values": scaled_fill_values,
        "raw_fill_values": raw_fill_values,
        "feature_clip_z": _FEATURE_CLIP_Z,

        "train_mean_return": baseline_mean,
        "best_model": best_model,
        "active_model_names": model_names,
        "model_pipeline_config": dict(MODEL_PIPELINE_CONFIG),
        "cv_history": history,
        "hyperparameter_tuning": tuning,
        "hyperparameter_tuning_enabled": ENABLE_HYPERPARAMETER_TUNING,
        "selected_model_configs": selected_configs,
        "huber_fixed_params": {
            "epsilon": _HUBER_EPS,
            "alpha": _HUBER_A,
        },
        "training_time_sec": train_elapsed,
    }

    for nm in model_names:
        params[f"{nm}_coef"] = np.asarray(fitted[nm].coef_, float)
        params[f"{nm}_intercept"] = _as_scalar_intercept(fitted[nm].intercept_)

    return params


def modelForecast(testFilename: str, parameters: dict) -> np.ndarray:
    infer_start = perf_counter()
    print("--------->Starting Forecasting...")

    if not isinstance(parameters, dict):
        raise ValueError("parameters must be a dict returned by modelEstimate().")

    model_names = list(parameters.get("active_model_names", _active_model_names()))
    if not model_names:
        raise ValueError("No active models found in parameters.")

    price_adjustments = parameters["price_adjustments"]
    df_test, _ = _build_frame(testFilename, price_adjustments=price_adjustments)

    feature_cols = parameters.get("feature_cols", FEATURE_COLS)

    X_test_raw_df = df_test[feature_cols].copy()
    X_test_raw_filled_df = X_test_raw_df.fillna(parameters["raw_fill_values"])

    X_test_scaled_df = _apply_feature_scaler(
        X_test_raw_filled_df,
        parameters["norm_mean"],
        parameters["norm_std"],
        fill_values=parameters.get("scaled_fill_values"),
        clip_z=parameters.get("feature_clip_z", _FEATURE_CLIP_Z),
    )
    X_test_scaled = X_test_scaled_df.to_numpy(dtype=float)

    close_mask = df_test["is_last_row_of_day"].to_numpy(dtype=bool)

    preds = {}
    for nm in model_names:
        preds[nm] = _linear_predict_from_params(parameters, nm, X_test_scaled, close_mask)

    _check_prediction_integrity(preds, close_mask, model_names=model_names, label="TEST")

    chosen_model = parameters.get("best_model", model_names[0])
    if chosen_model not in model_names:
        chosen_model = model_names[0]

    if _TARGET_COL in df_test.columns and df_test[_TARGET_COL].notna().any():
        score_mask = df_test[_TARGET_COL].notna().to_numpy()
        y_true = df_test.loc[score_mask, _TARGET_COL].to_numpy(dtype=float)
        baseline = float(parameters.get("train_mean_return", 0.0))
        baseline_mspe = _mse(y_true, np.full(len(y_true), baseline))

        bar = "=" * 100
        print(f"\n{bar}")
        print("TEST-SET METRICS - ACTIVE MODELS")
        print(f"Baseline MSPE (training mean = {baseline:.6f}): {baseline_mspe:.8f}")
        hdr = f"{'Model':<18}{'Test R²':>12}{'Test MSPE':>14}{'vs Baseline':>15}"
        print(hdr)
        print("-" * len(hdr))

        rows = []
        for nm in model_names:
            p = preds[nm][score_mask]
            mspe = _mse(y_true, p)
            rows.append((nm, _r2(y_true, p), mspe, _gain(mspe, baseline_mspe)))
            print(
                f"{nm:<18}"
                f"{_fmt_float(rows[-1][1], 6):>12}"
                f"{_fmt_float(mspe, 8):>14}"
                f"{_fmt_pct(rows[-1][3]):>15}"
            )

        print(f"\n{bar}")
        print("TEST LEADERBOARD — ranked by Test MSPE (lower is better)")
        print(hdr)
        print("-" * len(hdr))
        rows.sort(key=lambda x: x[2])
        for nm, r2v, mspe, gain in rows:
            print(
                f"{nm:<18}"
                f"{_fmt_float(r2v, 6):>12}"
                f"{_fmt_float(mspe, 8):>14}"
                f"{_fmt_pct(gain):>15}"
            )
        chosen_model = rows[0][0]
        print(f"\nBest test-phase model by MSPE: {chosen_model}")
        print(bar + "\n")
    else:
        print(f"Test file has no usable '{_TARGET_COL}' column — using best CV model: {chosen_model}")

    final_pred = preds[chosen_model]
    infer_elapsed = perf_counter() - infer_start
    parameters["inference_time_sec"] = infer_elapsed

    print(f"Final prediction returned : {chosen_model}")
    print(f"Predictions generated     : shape={final_pred.shape}, dtype={final_pred.dtype}")
    if final_pred.size > 0:
        print(f"Preview                   : {final_pred[:min(5, final_pred.size)]}")
    print(f"Inference time: {_fmt_seconds(infer_elapsed)}")

    return final_pred


# ============================================================
# Main execution block
# ============================================================
if __name__ == "__main__":
    script_start = perf_counter()

    TRAIN_CSV = "/Users/mazin/Desktop/super prep materials/GSAPred/Two-financial-instruments/train1.csv"
    TEST_CSV = "/Users/mazin/Desktop/super prep materials/GSAPred/Two-financial-instruments/test1.csv"

    params = modelEstimate(TRAIN_CSV)
    predictions = modelForecast(TEST_CSV, params)

    total_elapsed = perf_counter() - script_start
    training_elapsed = params.get("training_time_sec", float("nan"))
    inference_elapsed = params.get("inference_time_sec", float("nan"))

    bar = "=" * 100
    print(f"\n{bar}")
    print("EXECUTION TIME SUMMARY")
    print(bar)
    print(f"Training time : {_fmt_seconds(training_elapsed)}")
    print(f"Inference time: {_fmt_seconds(inference_elapsed)}")
    print(f"Total script  : {_fmt_seconds(total_elapsed)}")
    print(bar)

--------->Start Estimation...
Active models: ['Ridge']
Model pipeline config: {'Ridge': {'include': True, 'tune': False}, 'Lasso': {'include': False, 'tune': False}, 'Ridge_w': {'include': False, 'tune': False}, 'Lasso_w': {'include': False, 'tune': False}, 'Huber': {'include': False, 'tune': False}}
Global hyperparameter tuning enabled: False
DYNAMIC PRICE ADJUSTMENTS FROM TRAIN SET
xprice minimum                : 127.5375
xprice minimum minus 1        : 126.5375
xprice rounded adjustment used: 127.0

yprice minimum                : 147.175
yprice minimum minus 1        : 146.175
yprice rounded adjustment used: 146.0

----->Building cached CV folds...
----->Hyperparameter tuning globally disabled. Using preset configurations only.

HYPERPARAMETER SWITCH DECISIONS
Switch to tuned only if tuned mean validation improvement beats preset mean validation improvement.
Models with include=False are excluded before this stage.
Family            Tune Allowed?   Switched?    Preset Mean Δ%     T